In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1

from datasets import load_dataset
mnli = load_dataset("nyu-mll/multi_nli")

In [2]:

%%time
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# now for a subsample of the whole corpus since its too much like 400.000 examples or something. chose 5.000 for building. since otherwise im waiting for too long.

n = 5000
sample_train = mnli["train"].shuffle(seed=0).select(range(n))
print(sample_train)

# Put premise and Hypothesis together in one string for the tokeniser.
texts = [p + " " + h for p, h in zip(sample_train["premise"], sample_train["hypothesis"])]

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(texts, sample_train["label"])), 
            k=num_folds,
            seed=seed
        )
    )

Dataset({
    features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
    num_rows: 5000
})
CPU times: user 224 ms, sys: 2.65 ms, total: 227 ms
Wall time: 228 ms


In [3]:
from collections import Counter
Counter(sample_train["label"])
# label distribution still 1/3 each so went well.

Counter({2: 1674, 1: 1665, 0: 1661})

In [4]:
# Now lets do the tokeniser.
# first take all the premise and hypothesis of the train set.

all_train_texts = [p + " " + h for p, h in zip(mnli["train"]["premise"], mnli["train"]["hypothesis"])]

# choose same vocab size as in tutorial; bc its smaller and so a bit faster.
vocab_size = 1000
# min freq of a token to merge for bpe.
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(all_train_texts,vocab_size,min_frequency)
results_folder = Path.cwd().parent / "results"
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

In [5]:
vocab = tokenizer.get_vocab()
for _, (t, i) in zip(range(10), vocab.items()):
    print(f"{t} -> {i}")

ru -> 557
▁ke -> 909
¡ -> 94
gr -> 563
▁from -> 329
▁them -> 429
en -> 172
▁they're -> 904
é -> 110
▁V -> 546


In [6]:
print(all_train_texts[0])
print(tokenizer.encode(all_train_texts[0]).tokens)

Conceptually cream skimming has two basic dimensions - product and geography. Product and geography are what make cream skimming work. 
['▁C', 'on', 'ce', 'pt', 'ually', '▁c', 're', 'am', '▁s', 'k', 'im', 'm', 'ing', '▁has', '▁two', '▁bas', 'ic', '▁d', 'im', 'ens', 'ions', '▁', '-', '▁produ', 'ct', '▁and', '▁g', 'e', 'og', 'ra', 'p', 'h', 'y.', '▁P', 'ro', 'du', 'ct', '▁and', '▁g', 'e', 'og', 'ra', 'p', 'h', 'y', '▁are', '▁what', '▁make', '▁c', 're', 'am', '▁s', 'k', 'im', 'm', 'ing', '▁work', '.', '▁']


In [7]:
token_indicator_encoder = part1.clslib.TokenIndicator(tokenizer)
print(len(token_indicator_encoder))

1000
